# 06 – Model Testing: Radiance Prediction Demo

Fits a LightGBM model on older radiance / impervious-surface data for
**North Carolina** and **Utah**, then predicts radiance at a later date
and maps the county-level net mismatch between the prediction and
observed later radiance.

**Edit the configuration cell below** to point at your data and set the
date labels that will appear in figure titles.

In [ ]:
# =====================================================================
# CONFIGURATION – update to match your data before running
# =====================================================================

# Human-readable labels used in figure titles.
# They do NOT need to match the actual filenames; they are purely cosmetic.
OLDER_DATE = "2016"
LATER_DATE = "2021"

# Spatial neighbourhood window fed to the model (must be odd)
WINDOW_SIZE = 3

In [ ]:
import numpy as np
import xarray as xr
import rioxarray
import geopandas as gpd
import matplotlib.pyplot as plt
import lightgbm as lgb
from pathlib import Path
from rasterio.enums import Resampling
from numpy.lib.stride_tricks import sliding_window_view
from geocube.api.core import make_geocube
import config

# ── Paths (driven by config.py) ──────────────────────────────────────
STATES_SHP    = config.STATES_SHP
COUNTIES_SHP  = config.COUNTIES_SHP
RADIANCE_OLDER = config.RADIANCE_OLDER
RADIANCE_LATER = config.RADIANCE_NEWER
IMPERV_OLDER   = config.IMPERV_OLDER
IMPERV_LATER   = config.IMPERV_NEWER

In [ ]:
# =====================================================================
# HELPER FUNCTIONS
# =====================================================================

def load_and_match(path, target_da):
    """Clip and resample *path* raster to match the grid of *target_da*."""
    da = rioxarray.open_rasterio(path).squeeze()
    minx, miny, maxx, maxy = target_da.rio.bounds()
    da_clipped = da.rio.clip_box(minx, miny, maxx, maxy, crs=target_da.rio.crs)
    if da.rio.nodata is not None:
        da_clipped = da_clipped.where(da_clipped != da.rio.nodata)
    return da_clipped.rio.reproject_match(target_da, resampling=Resampling.average)


def create_spatial_features(data_array, window=3):
    """Convert a 2-D raster into a 2-D array of flattened spatial windows."""
    pad_size = window // 2
    filled = np.nan_to_num(data_array.values, nan=0.0)
    padded = np.pad(filled, pad_width=pad_size, mode='reflect')
    windows = sliding_window_view(padded, window_shape=(window, window))
    H, W = data_array.shape
    return windows.reshape(H * W, window * window)


def run_state_pipeline(state_name):
    """
    Full LightGBM pipeline for *state_name*.

    Steps
    -----
    1. Load older VIIRS radiance clipped to the state.
    2. Load and resample older + later impervious surfaces.
    3. Train LightGBM on (older impervious → older radiance).
    4. Predict radiance at later date using later impervious surface.
    5. Load observed later VIIRS radiance.
    6. Compute pixel-level mismatch = predicted_later − observed_later.

    Returns
    -------
    mismatch_da : xarray.DataArray
        Pixel-level (predicted later − observed later) radiance.
    viirs_later_da : xarray.DataArray
        Observed later radiance clipped to the state.
    state_gdf : geopandas.GeoDataFrame
        State polygon in the raster CRS.
    """
    print(f"\n{'='*60}")
    print(f"  Running pipeline: {state_name}")
    print(f"{'='*60}")

    # ------------------------------------------------------------------
    # 1. Load older VIIRS and clip to state
    # ------------------------------------------------------------------
    print(f"[{state_name}] Loading older ({OLDER_DATE}) VIIRS radiance...")
    viirs_older_da = rioxarray.open_rasterio(RADIANCE_OLDER).squeeze()
    if viirs_older_da.rio.nodata is not None:
        viirs_older_da = viirs_older_da.where(viirs_older_da != viirs_older_da.rio.nodata)

    states_gdf = gpd.read_file(STATES_SHP)
    state_gdf = states_gdf[states_gdf['NAME'] == state_name].to_crs(viirs_older_da.rio.crs)
    if state_gdf.empty:
        raise ValueError(f"State '{state_name}' not found in {STATES_SHP}")

    minx, miny, maxx, maxy = state_gdf.total_bounds
    viirs_older_da = viirs_older_da.rio.clip_box(minx, miny, maxx, maxy)
    viirs_older_da.load()
    print(f"[{state_name}] Older grid shape: {viirs_older_da.shape}")

    # ------------------------------------------------------------------
    # 2. Build a strict within-state mask
    # ------------------------------------------------------------------
    state_gdf = state_gdf.copy()
    state_gdf['STATEFP_int'] = state_gdf['STATEFP'].astype(int)
    state_mask_da = make_geocube(
        vector_data=state_gdf,
        measurements=['STATEFP_int'],
        like=viirs_older_da,
        fill=0,
    ).STATEFP_int
    master_mask = (state_mask_da > 0).values

    # ------------------------------------------------------------------
    # 3. Load and resample impervious surfaces
    # ------------------------------------------------------------------
    print(f"[{state_name}] Loading impervious surfaces...")
    imp_older_da = load_and_match(IMPERV_OLDER, viirs_older_da)
    imp_later_da = load_and_match(IMPERV_LATER, viirs_older_da)

    # ------------------------------------------------------------------
    # 4. Build training dataset
    # ------------------------------------------------------------------
    valid_mask_2d = (
        master_mask
        & ~np.isnan(viirs_older_da.values)
        & ~np.isnan(imp_older_da.values)
    )
    valid_mask_1d = valid_mask_2d.flatten()

    X_older = create_spatial_features(imp_older_da, WINDOW_SIZE)
    y_older = viirs_older_da.values.flatten()
    X_train = X_older[valid_mask_1d]
    y_train = y_older[valid_mask_1d]
    print(f"[{state_name}] Training set: {X_train.shape[0]:,} pixels, "
          f"{X_train.shape[1]} features")

    # ------------------------------------------------------------------
    # 5. Train LightGBM
    # ------------------------------------------------------------------
    print(f"[{state_name}] Training LightGBM...")
    model = lgb.LGBMRegressor(
        n_estimators=100,
        max_depth=15,
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    )
    model.fit(X_train, y_train)

    # ------------------------------------------------------------------
    # 6. Predict radiance at later date
    # ------------------------------------------------------------------
    print(f"[{state_name}] Predicting radiance at later date ({LATER_DATE})...")
    X_later = create_spatial_features(imp_later_da, WINDOW_SIZE)
    pred_later_2d = model.predict(X_later).reshape(viirs_older_da.shape)
    pred_later_2d = np.where(master_mask, pred_later_2d, np.nan)

    pred_later_da = xr.DataArray(
        pred_later_2d,
        coords=viirs_older_da.coords,
        dims=viirs_older_da.dims,
    )
    pred_later_da.rio.write_crs(viirs_older_da.rio.crs, inplace=True)

    # ------------------------------------------------------------------
    # 7. Load observed later VIIRS
    # ------------------------------------------------------------------
    print(f"[{state_name}] Loading observed later ({LATER_DATE}) VIIRS radiance...")
    viirs_later_da = rioxarray.open_rasterio(RADIANCE_LATER).squeeze()
    if viirs_later_da.rio.nodata is not None:
        viirs_later_da = viirs_later_da.where(viirs_later_da != viirs_later_da.rio.nodata)
    viirs_later_da = viirs_later_da.rio.clip_box(minx, miny, maxx, maxy)
    viirs_later_da = viirs_later_da.rio.reproject_match(viirs_older_da)
    viirs_later_da = viirs_later_da.where(master_mask)

    # ------------------------------------------------------------------
    # 8. Compute mismatch: predicted_later − observed_later
    # ------------------------------------------------------------------
    mismatch_da = pred_later_da - viirs_later_da

    print(f"[{state_name}] Pipeline complete.")
    return mismatch_da, viirs_later_da, state_gdf

In [ ]:
def plot_county_choropleth(mismatch_da, state_name, counties_shp, state_fips, title=None):
    """
    Aggregate pixel-level mismatch to county polygons and plot a choropleth.

    Parameters
    ----------
    mismatch_da : xarray.DataArray
        Pixel-level (predicted_later − observed_later) radiance.
    state_name : str
        Human-readable state name (used in the default title).
    counties_shp : path-like
        Path to the US counties shapefile.
    state_fips : str
        Two-digit state FIPS code string (e.g. '37' for NC, '49' for UT).
    title : str or None
        Override the auto-generated figure title.

    Returns
    -------
    geopandas.GeoDataFrame
        Counties with a 'mean_mismatch' column added.
    """
    if title is None:
        title = (
            f"{state_name}: Net Radiance Mismatch by County\n"
            f"(Predicted {LATER_DATE} − Observed {LATER_DATE}, nW/cm²/sr)"
        )

    # Load counties for this state
    counties = gpd.read_file(counties_shp)
    state_counties = counties[counties['STATEFP'] == state_fips].copy()
    state_counties = state_counties.to_crs(mismatch_da.rio.crs)

    # Rasterize county integer FIPS onto the mismatch grid
    state_counties['COUNTYFP_int'] = state_counties['COUNTYFP'].astype(int)
    county_id_da = make_geocube(
        vector_data=state_counties,
        measurements=['COUNTYFP_int'],
        like=mismatch_da,
        fill=0,
    ).COUNTYFP_int

    # Compute mean mismatch per county
    county_ids = county_id_da.values.flatten()
    mismatch_vals = mismatch_da.values.flatten()

    county_means = {}
    for fips_int in state_counties['COUNTYFP_int'].unique():
        mask = (county_ids == fips_int) & np.isfinite(mismatch_vals)
        county_means[fips_int] = mismatch_vals[mask].mean() if mask.sum() > 0 else np.nan

    state_counties['mean_mismatch'] = state_counties['COUNTYFP_int'].map(county_means)

    # Symmetric colour scale centred on zero
    abs_max = state_counties['mean_mismatch'].abs().max()

    fig, ax = plt.subplots(figsize=(12, 8), dpi=150)
    state_counties.plot(
        column='mean_mismatch',
        ax=ax,
        cmap='coolwarm',
        vmin=-abs_max,
        vmax=abs_max,
        legend=True,
        legend_kwds={'label': 'Mean mismatch (nW/cm²/sr)', 'shrink': 0.7},
        missing_kwds={'color': 'lightgrey', 'label': 'No data'},
        edgecolor='black',
        linewidth=0.3,
    )
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

    return state_counties

---
## North Carolina

In [ ]:
nc_mismatch_da, nc_viirs_later_da, nc_state_gdf = run_state_pipeline("North Carolina")

In [ ]:
nc_counties = plot_county_choropleth(
    nc_mismatch_da,
    state_name="North Carolina",
    counties_shp=COUNTIES_SHP,
    state_fips='37',
)

In [ ]:
# --- North Carolina mismatch summary ---
nc_obs_total   = float(nc_viirs_later_da.sum().values)
nc_net_error   = float(nc_mismatch_da.sum().values)
nc_mae         = float(np.abs(nc_mismatch_da).mean().values)

print(f"=== North Carolina – Radiance Mismatch ({LATER_DATE}) ===")
print(f"Total Observed Sum of Lights : {nc_obs_total:>15,.2f} nW/cm²/sr")
print(f"Net Mismatch (Predicted−Obs) : {nc_net_error:>+15,.2f} nW/cm²/sr "
      f"({nc_net_error / nc_obs_total * 100:+.2f}%)")
print(f"Mean Absolute Error per pixel: {nc_mae:>15.3f} nW/cm²/sr")

---
## Utah

In [ ]:
ut_mismatch_da, ut_viirs_later_da, ut_state_gdf = run_state_pipeline("Utah")

In [ ]:
ut_counties = plot_county_choropleth(
    ut_mismatch_da,
    state_name="Utah",
    counties_shp=COUNTIES_SHP,
    state_fips='49',
)

In [ ]:
# --- Utah mismatch summary ---
ut_obs_total   = float(ut_viirs_later_da.sum().values)
ut_net_error   = float(ut_mismatch_da.sum().values)
ut_mae         = float(np.abs(ut_mismatch_da).mean().values)

print(f"=== Utah – Radiance Mismatch ({LATER_DATE}) ===")
print(f"Total Observed Sum of Lights : {ut_obs_total:>15,.2f} nW/cm²/sr")
print(f"Net Mismatch (Predicted−Obs) : {ut_net_error:>+15,.2f} nW/cm²/sr "
      f"({ut_net_error / ut_obs_total * 100:+.2f}%)")
print(f"Mean Absolute Error per pixel: {ut_mae:>15.3f} nW/cm²/sr")